# Fusion Oncology: AI-Powered Cancer Target Discovery

[![GitHub](https://img.shields.io/badge/GitHub-fusion__oncology-blue)](https://github.com/mytechnotalent/fusion_oncology)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

## Overview

This notebook demonstrates **Fusion Oncology** on the **GDSC dataset** - combining:

- **XGBoost** feature importance (which genes discriminate cancer types)
- **DNABERT-2** sequence embeddings (structural gene fragility)
- **Multi-omics** integration (mutations, CNAs, methylation)
- **Clinical evidence** aggregation (OpenTargets, CIViC, ClinicalTrials.gov)
- **Drug-target** mapping and resistance prediction
- **Digital twin** tumor simulations

## Dataset

**GDSC: Genomics of Drug Sensitivity in Cancer**
- **Source**: 1,002 cancer cell lines with genomic features
- **Add to Kaggle**: [GDSC Dataset](https://www.kaggle.com/datasets/samiraalipour/genomics-of-drug-sensitivity-in-cancer-gdsc)
- **Input**: `/kaggle/input/genomics-of-drug-sensitivity-in-cancer-gdsc/`
- **Output**: `/kaggle/working/`
- **Note**: Cell lines are lab-adapted models for workflow demonstration

## Step 1: Setup and Installation

In [ ]:
# Install Fusion Oncology (compatible with Kaggle's pre-installed packages)
%pip install -q "openpyxl>=3.1,<4" "fusion-oncology @ git+https://github.com/mytechnotalent/fusion_oncology.git@main"
print("Installation complete!")

In [ ]:
# Import all libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Fusion Oncology imports
from fusion_oncology.config import ProjectConfig
from fusion_oncology.data.ingestion import DataIngestion
from fusion_oncology.models.fusion import FusionEngine
from fusion_oncology.analysis.clinical_evidence import ClinicalEvidenceAggregator
from fusion_oncology.analysis.drug_target import DrugTargetMapper
from fusion_oncology.analysis.resistance import ResistancePredictor
from fusion_oncology.models.digital_twin import DigitalTwin, SimulationConfig, DrugRegimen
from fusion_oncology.models.companion_dx import PatientProfile, CompanionDiagnostic
from fusion_oncology.analysis.pathway import PathwayEnrichment

## Step 2: Load GDSC Dataset

Loading cancer cell line data with genomic features and tissue classifications.

In [ ]:
# Configure project settings (output to /kaggle/working/)
cfg = ProjectConfig(
    top_k_genes=5,
    fuzz_iterations=10,
    xgb_n_estimators=50,
    output_dir=Path("/kaggle/working/results"),
)
print(f"Output directory: /kaggle/working/results")

In [ ]:
# GDSC Input paths (exact file names from the Kaggle dataset):
GDSC_DIR = "/kaggle/input/genomics-of-drug-sensitivity-in-cancer-gdsc"
cell_lines = f"{GDSC_DIR}/Cell_Lines_Details.xlsx"
gdsc_main = f"{GDSC_DIR}/GDSC_DATASET.csv"
gdsc2 = f"{GDSC_DIR}/GDSC2-dataset.csv"
compounds = f"{GDSC_DIR}/Compounds-annotation.csv"

print("GDSC dataset files:")
print(f"  1. GDSC_DATASET.csv      (main merged dataset)")
print(f"  2. GDSC2-dataset.csv     (raw drug sensitivity, IC50)")
print(f"  3. Cell_Lines_Details.xlsx (cell line metadata)")
print(f"  4. Compounds-annotation.csv (drug targets/pathways)")

In [ ]:
# Load all 4 GDSC files
df_main = pd.read_csv(gdsc_main)
df_gdsc2 = pd.read_csv(gdsc2)
df_compounds = pd.read_csv(compounds)
df_cellinfo = pd.read_excel(cell_lines, sheet_name="Cell line details")
print("=== GDSC_DATASET.csv (main merged file) ===")
print(f"Shape: {df_main.shape[0]} rows x {df_main.shape[1]} columns")
print(f"Columns: {', '.join(df_main.columns.tolist())}\n")
print("=== GDSC2-dataset.csv (raw drug sensitivity) ===")
print(f"Shape: {df_gdsc2.shape[0]} rows x {df_gdsc2.shape[1]} columns")
print(f"Columns: {', '.join(df_gdsc2.columns.tolist())}\n")
print("=== Compounds-annotation.csv (drug info) ===")
print(f"Shape: {df_compounds.shape[0]} rows x {df_compounds.shape[1]} columns")
print(f"Columns: {', '.join(df_compounds.columns.tolist())}\n")
print("=== Cell_Lines_Details.xlsx (cell line metadata) ===")
print(f"Shape: {df_cellinfo.shape[0]} rows x {df_cellinfo.shape[1]} columns")
print(f"Columns: {', '.join(df_cellinfo.columns.tolist())}\n")

# Extract features (X) and labels (y) from the main merged dataset
if "TCGA_DESC" in df_main.columns:
    y = df_main["TCGA_DESC"].dropna()
    numeric_cols = df_main.select_dtypes(include=[np.number]).columns.tolist()
    X = df_main.loc[y.index, numeric_cols].fillna(0)
    print(f"Features: {len(numeric_cols)} numeric columns from GDSC_DATASET.csv")
    print(f"Labels: TCGA_DESC ({y.nunique()} cancer types, {len(y)} samples)")
else:
    print("TCGA_DESC not found, using built-in UCI dataset as fallback...")
    ingestor = DataIngestion(cfg)
    X, y = ingestor.get_patient_data()
    print(f"Loaded {X.shape[0]} samples x {X.shape[1]} features (UCI fallback)")

## Step 3: Exploratory Data Analysis

In [ ]:
# Cancer type distribution
fig, ax = plt.subplots(figsize=(10, 6))
y.value_counts().plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Cancer Type Distribution", fontsize=14, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
# Top variable features
top_features = X.var().nlargest(50).index
sample_subset = X.sample(n=100, random_state=42) if len(X) > 100 else X
print(f"Most variable features: {', '.join(str(c) for c in top_features[:10])}")

In [ ]:
# Feature variance heatmap
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    sample_subset[top_features].T, cmap="RdBu_r", center=0, cbar_kws={"label": "Value"}, ax=ax
)
ax.set_title("Top 50 Variable Features (GDSC)", fontsize=14, fontweight="bold")
plt.show()

In [ ]:
# GDSC drug targets and pathways from Compounds-annotation.csv
print("=== Drug Targets (from Compounds-annotation.csv) ===")
if "TARGET_PATHWAY" in df_compounds.columns:
    print(f"Unique drugs: {df_compounds['DRUG_NAME'].nunique()}")
    print(f"Unique targets: {df_compounds['TARGET'].nunique()}")
    print(f"\nTop target pathways:")
    print(df_compounds["TARGET_PATHWAY"].value_counts().head(10).to_string())
elif "PATHWAY_NAME" in df_compounds.columns:
    print(df_compounds["PATHWAY_NAME"].value_counts().head(10).to_string())

In [ ]:
# Drug sensitivity distribution from GDSC2
if "LN_IC50" in df_gdsc2.columns:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.hist(df_gdsc2["LN_IC50"].dropna(), bins=50, color="steelblue")
    ax.set_title("LN_IC50 Distribution", fontsize=14, fontweight="bold")
    ax.set_xlabel("LN_IC50 (lower = more sensitive)")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
    fig, ax = plt.subplots(figsize=(12, 6))
    top_drugs = df_gdsc2["DRUG_NAME"].value_counts().head(15)
    ax.barh(top_drugs.index, top_drugs.values, color="coral")
    ax.set_title("Most Tested Drugs", fontsize=14, fontweight="bold")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

## Step 4: Run Fusion Analysis

This combines:
1. **XGBoost** - Train multi-class classifier and extract feature importance
2. **DNABERT-2** - Compute sequence instability scores via mutation fuzzing
3. **Fusion Index** - Importance x Instability x 1000 (higher = better target)
4. **Pathway enrichment** - Map to cancer pathways
5. **Drug annotation** - Link to approved therapies

In [ ]:
# Run fusion analysis
print("Running Fusion Analysis...\n")
engine = FusionEngine(cfg)
results = engine.run(X, y)
print("Analysis complete!")

In [ ]:
# Display top targets
print("\n" + "=" * 60)
print("TOP THERAPEUTIC TARGETS")
print("=" * 60)
print(results[["Gene", "Fusion_Index"]].to_string(index=False))
print("=" * 60)

## Step 5: Clinical Evidence and Drug Mapping

In [ ]:
# Get top gene
top_gene = results.iloc[0]["Gene"]
print(f"Clinical Evidence for {top_gene}:")

In [ ]:
# Query evidence sources
agg = ClinicalEvidenceAggregator(cfg)
evidence = agg.profile(top_gene)

In [ ]:
# Display evidence scores
print(f"  OpenTargets: {evidence.get('opentargets', {}).get('overall_score', 0):.3f}")
print(f"  Clinical Trials: {len(evidence.get('trials', []))}")
print(f"  CIViC Evidence: {len(evidence.get('civic', []))}")
print(f"  Composite Score: {evidence.get('evidence_score', 0):.3f}")

In [ ]:
# Map to available drugs
mapper = DrugTargetMapper(cfg)
drug_df = mapper.annotate(results)

In [ ]:
# Display drug targets
print(f"\nDrug Targets for {top_gene}:")
drug_matches = drug_df[drug_df["Gene"] == top_gene]["Drugs"].iloc[0]
if drug_matches and drug_matches != "None":
    for drug in drug_matches.split(", "):
        print(f"  - {drug}")

## Step 6: Resistance Mechanisms

In [ ]:
# Query resistance mechanisms
predictor = ResistancePredictor(cfg)
resistance_report = predictor.full_report([top_gene])

In [ ]:
# Display resistance info
print(f"Resistance Mechanisms for {top_gene}:")
if not resistance_report.empty:
    for _, row in resistance_report.iterrows():
        print(f"  {row['Mechanism']} - {row['Risk_Level']}")
else:
    print("  No known resistance")

## Step 7: Digital Twin Tumor Simulation

In [ ]:
# Initialize digital twin
sim_cfg = SimulationConfig(simulation_days=180)
twin = DigitalTwin(sim_config=sim_cfg, project_config=cfg)

In [ ]:
# Add treatment regimen
twin.add_regimen(
    DrugRegimen(name="Targeted Therapy", efficacy=0.15, resistance_rate=0.001, duration_days=180)
)

In [ ]:
# Run simulation
trajectory = twin.simulate()
summary = twin.summary()
print(f"RECIST Response: {summary['recist']}")

In [ ]:
# Plot tumor volume
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(trajectory["day"], trajectory["total"], color="darkred", linewidth=2)
ax.set_title("Tumor Volume Over Time", fontweight="bold")
ax.set_yscale("log")
plt.show()

In [ ]:
# Plot cell populations
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(trajectory["day"], trajectory["sensitive"], label="Sensitive", color="green")
ax.plot(trajectory["day"], trajectory["resistant"], label="Resistant", color="red")
ax.set_yscale("log")
ax.legend()
plt.show()

In [ ]:
# Display simulation results
print(f"Best Response: {summary['best_response']['response_pct']:.1f}% reduction")
print(f"Final Volume: {summary['final_tumour']:.2e} mm3")

## Step 8: Companion Diagnostics (Patient-Specific)

In [ ]:
# Create example patient profile (using GDSC tissue type)
patient = PatientProfile(
    patient_id="GDSC-001",
    cancer_type=str(y.iloc[0]),  # Use actual GDSC tissue classification
    mutations=[{"gene": "EGFR", "variant": "L858R", "vaf": 0.42}],
)

In [ ]:
# Generate treatment report
dx = CompanionDiagnostic(cfg)
report = dx.generate_report(patient)

In [ ]:
# Display top recommendations
print("TOP RECOMMENDATIONS:")
for i, rec in enumerate(report["recommendations"][:3], 1):
    print(f"{i}. {rec['drug']} - {rec['confidence']:.1%} confidence")
    print(f"   Tier: {rec['amp_asco_cap_tier']}")

## Step 9: Visualization Dashboard

In [ ]:
# Fusion Index ranking
results_sorted = results.sort_values("Fusion_Index", ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(results_sorted["Gene"], results_sorted["Fusion_Index"], color="steelblue")
ax.set_title("Top Targets", fontweight="bold")
plt.show()

In [ ]:
# Get pathway data
pathway_enricher = PathwayEnrichment(cfg)
pathway_df = pathway_enricher.annotate(results)

In [ ]:
# Count pathway occurrences
all_pathways = []
for pw_list in pathway_df["Pathways"]:
    if pw_list != "None":
        all_pathways.extend(pw_list.split(", "))
pathway_counts = pd.Series(all_pathways).value_counts().head(10)

In [ ]:
# Plot pathway enrichment
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(pathway_counts.index, pathway_counts.values, color="mediumseagreen")
ax.set_title("Top Cancer Pathways", fontweight="bold")
plt.show()
print("Dashboard complete!")

In [ ]:
# Drug availability
drug_counts = drug_df["Drugs"].apply(lambda x: 0 if x == "None" else len(x.split(", ")))
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(drug_df["Gene"], drug_counts, color="coral")
ax.set_title("Available Drugs", fontweight="bold")
plt.show()

In [ ]:
# Importance vs Instability
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    results["XGB_Importance"],
    results["Instability"],
    s=results["Fusion_Index"] * 200,
    c=results["Fusion_Index"],
    cmap="viridis",
    alpha=0.7,
)
plt.colorbar(scatter, label="Fusion Index")
plt.show()

## Step 10: Complete Inference Pipeline (End-to-End Demo)

**This cell runs the ENTIRE workflow on GDSC data:**
1. Load cell line sample
2. Run fusion analysis
3. Get clinical evidence
4. Map to drugs
5. Check resistance
6. Simulate treatment
7. Generate companion diagnostics report

In [ ]:
# Final summary introduction
print("=" * 70)
print("FUSION ONCOLOGY - COMPLETE INFERENCE PIPELINE")
print("=" * 70)

# STEP 1: Load and analyze cell line sample from GDSC
print("\n[1/7] Loading GDSC cell line sample...")
sample_cell_line = X.iloc[0:1]  # Get first cell line as example
true_tissue = y.iloc[0]
print(f"Cell line sample loaded | Tissue type: {true_tissue}")

# STEP 2: Run Fusion Analysis
print("\n[2/7] Running Fusion Analysis (XGBoost + DNABERT-2)...")
engine_demo = FusionEngine(cfg)
fusion_results = engine_demo.run(X, y)
top_3_genes = fusion_results.head(3)["Gene"].tolist()
print(f"Top 3 therapeutic targets: {', '.join(top_3_genes)}")

# STEP 3: Clinical Evidence Aggregation
print("\n[3/7] Querying clinical evidence databases...")
agg_demo = ClinicalEvidenceAggregator(cfg)
gene_evidence = agg_demo.profile(top_3_genes[0])
print(f"Evidence score: {gene_evidence.get('evidence_score', 0):.3f}")
print(f"  OpenTargets: {gene_evidence.get('opentargets', {}).get('overall_score', 0):.3f}")
print(f"  Clinical Trials: {len(gene_evidence.get('trials', []))}")

# STEP 4: Drug-Target Mapping
print("\n[4/7] Mapping to FDA-approved drugs...")
mapper_demo = DrugTargetMapper(cfg)
drug_results = mapper_demo.annotate(fusion_results)
top_drugs = drug_results.iloc[0]["Drugs"]
print(f"Available drugs: {top_drugs if top_drugs != 'None' else 'No direct matches'}")

# STEP 5: Resistance Prediction
print("\n[5/7] Analyzing resistance mechanisms...")
resist_demo = ResistancePredictor(cfg)
resist_report = resist_demo.full_report(top_3_genes)
print(f"Resistance risks identified: {len(resist_report)} mechanisms")

# STEP 6: Digital Twin Simulation
print("\n[6/7] Simulating treatment response (180 days)...")
twin_demo = DigitalTwin(sim_config=SimulationConfig(simulation_days=180), project_config=cfg)
twin_demo.add_regimen(
    DrugRegimen(name="Targeted Therapy", efficacy=0.15, resistance_rate=0.001, duration_days=180)
)
traj = twin_demo.simulate()
sim_summary = twin_demo.summary()
print(f"RECIST Response: {sim_summary['recist']}")
print(f"  Best response: {sim_summary['best_response']['response_pct']:.1f}% reduction")

# STEP 7: Companion Diagnostics Report
print("\n[7/7] Generating companion diagnostics...")
test_profile = PatientProfile(
    patient_id="GDSC-DEMO-001",
    cancer_type=str(true_tissue),
    mutations=[{"gene": top_3_genes[0], "variant": "V600E", "vaf": 0.35}],
)
dx_demo = CompanionDiagnostic(cfg)
dx_report = dx_demo.generate_report(test_profile)
print(f"Treatment recommendations: {len(dx_report.get('recommendations', []))}")

# FINAL OUTPUT
print("\n" + "=" * 70)
print("INFERENCE COMPLETE - All 7 pipeline stages executed successfully")
print("=" * 70)
print(f"\nSUMMARY:")
print(f"  Sample: GDSC cell line ({true_tissue})")
print(f"  Top Target: {top_3_genes[0]}")
print(f"  Evidence Score: {gene_evidence.get('evidence_score', 0):.3f}/1.0")
print(f"  Available Drugs: {top_drugs if top_drugs != 'None' else 'None'}")
print(f"  Predicted Response: {sim_summary['recist']}")
print(f"  Output saved to: /kaggle/working/results/")
print("\nReady for clinical validation and wet lab experiments")

---

## What This Notebook Delivers

### Complete Capabilities Demonstrated:

1. **Multi-Modal AI Fusion** - XGBoost feature importance (which genes matter most)
   - DNABERT-2 sequence instability (structural vulnerabilities)
   - Fusion Index = Importance x Instability x 1000

2. **Clinical Evidence Integration**
   - OpenTargets disease association scores
   - CIViC clinical interpretations
   - ClinicalTrials.gov active studies
   - PubMed literature citations

3. **Drug-Target Mapping**
   - FDA-approved therapies
   - Clinical trial drugs
   - Mechanism of action annotations
   - Druggability assessments

4. **Resistance Prediction**
   - Known resistance mutations
   - Compensatory pathway activation
   - Alternative splice variants
   - Risk level stratification

5. **Digital Twin Simulations**
   - 6-month treatment projections
   - Sensitive vs resistant cell populations
   - RECIST response categories
   - Best response timing

6. **Companion Diagnostics**
   - Patient-specific mutation profiling
   - AMP/ASCO/CAP tier classifications
   - Treatment recommendations with confidence scores
   - Personalized therapy selection

7. **Pathway Enrichment**
   - KEGG cancer pathway mapping
   - GO biological process terms
   - Hallmark gene sets

### Research Use Cases:

- **Discovery**: Identify novel therapeutic targets in your tumor cohort
- **Validation**: Prioritize candidates for CRISPR screens
- **Translation**: Map targets to existing FDA-approved drugs
- **Prediction**: Simulate treatment outcomes before clinical trials
- **Personalization**: Generate companion diagnostic reports

### Output Files (saved to `/kaggle/working/results/`):

- `fusion_results.csv` - Ranked gene targets with scores
- `clinical_evidence.json` - Aggregated evidence per gene
- `drug_annotations.csv` - Drug-target mappings
- `resistance_report.csv` - Known resistance mechanisms
- `digital_twin_trajectory.csv` - Simulation time series
- `companion_dx_report.json` - Patient treatment recommendations

---

### Important Disclaimer

**This is a research tool for hypothesis generation, NOT a clinical diagnostic device.**

- Requires experimental validation in lab (CRISPR, RNAi, drug screens)
- Does not replace expert clinical judgment
- Not FDA approved for clinical decision-making
- Results should be validated in independent cohorts
- Consult with oncologists and regulatory experts before clinical use

**Intended Users:** Cancer researchers, bioinformaticians, computational biologists, precision oncology teams

---

## Resources and Citation

**GitHub Repository:**  
[github.com/mytechnotalent/fusion_oncology](https://github.com/mytechnotalent/fusion_oncology)

**Documentation:**  
See README for full CLI reference and API documentation

**Citation:**
```bibtex
@software{fusion_oncology_2026,
  title={Fusion Oncology: Multi-Modal AI for Cancer Target Discovery},
  author={Thomas, Kevin},
  year={2026},
  url={https://github.com/mytechnotalent/fusion_oncology},
  note={Combines XGBoost, DNABERT-2, and clinical evidence for therapeutic target identification}
}
```

**License:** MIT | **Contact:** [@mytechnotalent](https://github.com/mytechnotalent)

---

**Questions or issues?** Open an issue on GitHub or contribute via pull request!